# Full-protein sequence and secondary-structure repeat boundaries

Audits source units against complete protein sequences and residue-level DSSP/author secondary structure. The real middle copy of the jointly supported primitive repeat is passed to HURDLER and codon optimization; structure-derived RepeatsDB units remain authoritative when a shorter natural harmonic lacks 3D confirmation.

**Rules:** `legacy-optimized-v1`; **seed:** 42 unless explicitly noted.

In [ ]:
REPO = '/home/wendai/projects/hurdler/clone_repeat_protein'
RULE_PROFILE = 'legacy-optimized-v1'
BOUNDARY_TABLE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_boundary_audit.parquet'
CANDIDATE_TABLE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_period_candidates.parquet'
UNIT_TABLE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_unit_alignment.parquet'
POSITION_TABLE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_position_variability.parquet'
SS_CANDIDATE_TABLE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_secondary_structure_candidates.parquet'
SS_RESIDUE_TABLE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_secondary_structure_residues.parquet'
VALIDATION = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/periodic_v4/module_boundary_validation.json'
FIGURE_DIR = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/figures/periodic_v4'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd

def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

run_context = {'rule_profile': RULE_PROFILE, 'input_hashes': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': []}

In [ ]:
boundary = pd.read_parquet(BOUNDARY_TABLE)
candidates = pd.read_parquet(CANDIDATE_TABLE)
units = pd.read_parquet(UNIT_TABLE)
positions = pd.read_parquet(POSITION_TABLE)
ss_candidates = pd.read_parquet(SS_CANDIDATE_TABLE)
ss_residues = pd.read_parquet(SS_RESIDUE_TABLE)
validation = json.loads(Path(VALIDATION).read_text())
run_context['input_hashes'] = {Path(path).name: sha256(path) for path in (BOUNDARY_TABLE, CANDIDATE_TABLE, UNIT_TABLE, POSITION_TABLE, SS_CANDIDATE_TABLE, SS_RESIDUE_TABLE, VALIDATION)}
run_context['row_counts'] = {'proteins': len(boundary), 'period_candidates': len(candidates), 'secondary_structure_candidate_scores': len(ss_candidates), 'secondary_structure_residues': len(ss_residues), 'aligned_units': len(units), 'module_positions': len(positions), 'natural': int(boundary.collection.eq('natural100').sum()), 'designed': int(boundary.collection.eq('designed_all').sum())}
run_context['filter_flow'] = ['read the complete protein/construct sequence', 'map RepeatsDB author chains exactly to RCSB label chains', 'run DSSP on natural and THR coordinates or use the DHR author H-loop-H-loop residue template', 'score every candidate independently by amino-acid Fourier/self-similarity and H/E/C state/transition periodicity', 'select the smallest harmonic passing both evidence gates', 'choose the real middle repeat copy, with an exact central tie resolved toward the earlier copy', 'pass that middle AA sequence to HURDLER and codon optimization', 'call positions fixed when conservation across inferred copies is at least 0.8']
run_context['limitations'] = ['a shorter natural harmonic is reported for review but does not replace a structure-derived RepeatsDB unit without 3D superposition', 'DHR secondary structure comes from the author residue-count template; THR and natural entries use DSSP', 'missing or mismatched structure chains are explicit failures, never inferred annotations']
validation

In [ ]:
summary = boundary.groupby('module_type').agg(proteins=('module_id','nunique'), median_source_length=('prior_unit_length','median'), median_primitive_length=('primitive_period','median'), split_source_units=('length_ratio', lambda values: int((values > 1.08).sum())), secondary_structure_passed=('secondary_structure_status', lambda values: int((values == 'passed').sum())), jointly_selected=('secondary_structure_selected_support', lambda values: int(values.fillna(False).sum())), manual_review=('qa_flags', lambda values: int((values != '').sum()))).reset_index()
summary

In [ ]:
boundary[['module_id','module_type','prior_unit_start','prior_unit_end','repeat_region_start','repeat_region_end','first_module_start','first_module_end','selected_module_index','selected_module_start','selected_module_end','unit_sequence','prior_unit_length','primitive_period','repeat_count','periodicity_score','secondary_structure_known_fraction','secondary_structure_selected_support','selection_reason','qa_flags']].sort_values(['module_type','module_id'])

In [ ]:
from IPython.display import Image, display
for name in ['source_vs_primitive_module_length.png','module_harmonic_ratio.png','module_fixed_fraction.png','secondary_structure_coverage.png','sequence_vs_secondary_structure_evidence.png','module_boundary_examples.png']:
    path = Path(FIGURE_DIR) / name
    assert path.stat().st_size > 0
    display(Image(filename=str(path)))

The colored interval is the inferred repeat region within the full protein. White ticks are module boundaries; the black interval is the source unit. The AA sequence used downstream is the real middle copy of that region, with an exact central tie resolved toward the earlier copy. Designed harmonics are accepted only when both amino-acid and residue-level secondary-structure periodicity pass. RepeatsDB's structure-derived natural boundary is retained unless later 3D superposition establishes a smaller complete unit.

In [ ]:
run_context